<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/05_construction_storytelling_visuals_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# จากข้อมูลจัดซื้อจัดจ้างสู่สัญญาที่ควรตรวจสอบก่อน

Notebook นี้เป็น **presentation layer** ของโครงการ DADS5001 โดยอ่านผลลัพธ์จาก Notebook 02–04 แล้วสร้าง visual ตามลำดับเรื่อง ไม่คำนวณนิยาม scope หรือ criteria ใหม่

ลำดับเรื่องมี 8 ขั้นตอน:

1. เตรียมข้อมูลจาก e-GP สู่ข้อมูลระดับสัญญา
2. สำรวจภาพรวมวงเงินและวิธีจัดซื้อของสัญญาก่อสร้าง
3. ขยายดูบริเวณ 500,000 บาท โดยแยกตามวิธีจัดซื้อ
4. กำหนดขอบเขตศึกษา Pattern จากข้อค้นพบ
5. วิเคราะห์ Pattern 1: สัญญาใกล้เพดานเกิดซ้ำ
6. วิเคราะห์ Pattern 2: การพึ่งพาผู้รับจ้างในหน่วยงาน
7. หาจุดตัดของ Pattern 1 และ Pattern 2
8. จัดลำดับคู่หน่วยงาน–ผู้รับจ้างที่ควรเปิดเอกสารก่อน

ผลลัพธ์เป็นสัญญาสำหรับตรวจสอบต่อ ไม่ใช่หลักฐานการทุจริตหรือการแบ่งซื้อแบ่งจ้าง


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from textwrap import shorten

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from matplotlib.patches import (
    Rectangle,
    FancyBboxPatch,
    FancyArrowPatch
)
from IPython.display import HTML, display


def display_table(data):
    html = data.to_html(
        index=False,
        border=0,
        classes='dataframe'
    )
    display(HTML(html))


# ดาวน์โหลดฟอนต์ภาษาไทยสำหรับ Matplotlib
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

fm.fontManager.addfont(
    'thsarabunnew-webfont.ttf'
)

pd.set_option(
    'display.max_columns',
    None
)
pd.set_option(
    'display.max_colwidth',
    100
)
pd.set_option(
    'display.float_format',
    lambda value: f'{value:,.2f}'
)

## 1. เปิดผลลัพธ์จาก Notebook 02–04

Notebook 05 ใช้ไฟล์ที่ผ่านการเตรียมข้อมูลและสร้าง flag แล้ว หากไฟล์ไม่ครบ ให้ Run all Notebook 02 → 03 → 04 ก่อน


In [ ]:
processed_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/'
    'egp-contract/processed'
)

figure_dir = processed_dir.parents[3] / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

contract_path = processed_dir / 'construction_contract_supplier_study_scope_2569.csv'
flags_path = processed_dir / 'construction_contract_review_indicators_2569.csv'
repeated_pairs_path = processed_dir / 'repeated_near_500k_agency_supplier_2569.csv'
dependence_path = processed_dir / 'agency_supplier_dependence_2569.csv'
priority_path = processed_dir / 'priority_review_contracts_2569.csv'

required_paths = [
    contract_path,
    flags_path,
    repeated_pairs_path,
    dependence_path,
    priority_path
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_text = '\n'.join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        'ไม่พบไฟล์ผลลัพธ์ต่อไปนี้ กรุณา Run all Notebook 02–04 ก่อน:\n'
        f'{missing_text}'
    )

contract_data = pd.read_csv(contract_path, low_memory=False)
contract_flags = pd.read_csv(flags_path, low_memory=False)
repeated_pairs = pd.read_csv(repeated_pairs_path, low_memory=False)
pattern2_pairs = pd.read_csv(dependence_path, low_memory=False)
priority_contracts = pd.read_csv(priority_path, low_memory=False)

file_summary = pd.DataFrame({
    'ข้อมูล': [
        'รายการสัญญา–ผู้รับจ้างก่อสร้าง',
        'รายการพร้อม flag',
        'คู่ที่เข้า Pattern 1',
        'คู่ที่เข้า Pattern 2',
        'รายการตรวจสอบลำดับแรก'
    ],
    'จำนวนแถว': [
        len(contract_data),
        len(contract_flags),
        len(repeated_pairs),
        len(pattern2_pairs),
        len(priority_contracts)
    ]
})

display_table(file_summary)


In [ ]:
project_id_column = 'รหัสโครงการ'
contract_column = 'เลขที่สัญญา'
contract_value_column = 'วงเงินงบประมาณในสัญญา (บาท)'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'
agency_column = 'ชื่อหน่วยงาน'
province_column = 'จังหวัด'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'
scope_column = 'อยู่ในขอบเขตตรวจรูปแบบ'

flag_columns = [
    'flag_pattern_1',
    'flag_pattern_2',
    'priority_review'
]

for column in flag_columns:
    contract_flags[column] = (
        contract_flags[column]
        .astype('boolean')
        .fillna(False)
    )

study_data = contract_data.loc[contract_data[scope_column]].copy()
near_ceiling_data = study_data.loc[
    study_data[contract_value_column].between(490000, 500000)
].copy()

print(f'รายการสัญญาก่อสร้างทั้งหมด: {len(contract_data):,}')
print(f'รายการในขอบเขตศึกษา: {len(study_data):,}')
print(f'รายการใกล้เพดาน: {len(near_ceiling_data):,}')
print(f'รายการตรวจสอบลำดับแรก: {len(priority_contracts):,}')


## 2. Visual theme

ใช้สีตามความหมายเดียวกันทุกภาพ:

- น้ำเงิน: ข้อมูลหลัก
- ส้ม: จุดใกล้เพดานหรือสิ่งที่ต้องสนใจ
- แดง: รายการตรวจสอบลำดับแรก
- เทา: ข้อมูลเปรียบเทียบ


In [ ]:
FIG_SIZE = (12, 6.75)
EXPORT_DPI = 120
COLORS = {
    'primary': '#365F7D',
    'secondary': '#6F8FA6',
    'highlight': '#D9822B',
    'risk': '#B5473C',
    'neutral': '#98A2B3',
    'light': '#E4E7EC',
    'text': '#344054',
    'muted': '#667085',
    'background': '#FFFFFF'
}

plt.rcParams.update({
    'font.family': 'TH Sarabun New',
    'font.size': 10,
    'text.color': COLORS['text'],
    'axes.labelcolor': COLORS['text'],
    'axes.titlecolor': COLORS['text'],
    'axes.titlesize': 15,
    'axes.titleweight': 'semibold',
    'axes.titlelocation': 'left',
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'xtick.color': COLORS['muted'],
    'ytick.color': COLORS['muted'],
    'axes.edgecolor': COLORS['light'],
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
    'grid.color': COLORS['light'],
    'grid.linewidth': 0.8,
    'figure.facecolor': 'white',
    'figure.dpi': 100,
    'savefig.dpi': EXPORT_DPI,
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'svg.fonttype': 'none',
    'axes.unicode_minus': False
})


def add_subtitle(ax, text):
    ax.text(
        0,
        1.01,
        text,
        transform=ax.transAxes,
        fontsize=9,
        color=COLORS['muted'],
        ha='left',
        va='bottom'
    )


def clean_axis(ax, grid_axis='x'):
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis=grid_axis, color=COLORS['light'])
    ax.grid(axis='y' if grid_axis == 'x' else 'x', visible=False)


def save_figure(fig, stem):
    png_path = figure_dir / f'{stem}.png'
    svg_path = figure_dir / f'{stem}.svg'

    fig.savefig(
        png_path,
        dpi=EXPORT_DPI,
        facecolor='white'
    )
    fig.savefig(
        svg_path,
        facecolor='white'
    )

    print(f'Saved: {png_path}')
    print(f'Saved: {svg_path}')


## Story 1 — เรามีข้อมูลอะไร และเตรียมข้อมูลระดับสัญญาอย่างไร

เริ่มจากข้อมูล e-GP ปีงบประมาณ 2569 จำนวน 3,964,924 ระเบียน เลือกเฉพาะสัญญาจ้างก่อสร้าง 180,079 สัญญา แล้วตัดสัญญาที่รหัสผู้รับจ้างขึ้นต้นด้วย `D` จำนวน 363 สัญญา เหลือข้อมูลวิเคราะห์ระดับสัญญา 179,716 สัญญา

หนึ่งสัญญากำหนดด้วย `รหัสโครงการ + รหัสผู้รับจ้าง + เลขที่สัญญา`


In [ ]:
source_record_count = 3964924
construction_contract_count = 180079
excluded_d_count = 363
analysis_contract_count = len(contract_data)
unique_project_count = contract_data[project_id_column].nunique()

fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.set_xlim(0, 15)
ax.set_ylim(0, 7)
ax.axis('off')


def add_data_box(x, y, width, height, title, value, unit, color):
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle='round,pad=0.03,rounding_size=0.16',
        linewidth=1.3,
        edgecolor=color,
        facecolor='white'
    )
    ax.add_patch(box)
    ax.text(
        x + width / 2,
        y + height * 0.67,
        title,
        ha='center',
        va='center',
        fontsize=9,
        color=COLORS['text']
    )
    ax.text(
        x + width / 2,
        y + height * 0.34,
        f'{value:,.0f} {unit}',
        ha='center',
        va='center',
        fontsize=14,
        fontweight='semibold',
        color=color
    )


def add_data_arrow(start, end, label):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle='-|>',
            mutation_scale=15,
            linewidth=1.4,
            color=COLORS['neutral']
        )
    )
    ax.text(
        (start[0] + end[0]) / 2,
        start[1] + 0.48,
        label,
        ha='center',
        va='bottom',
        fontsize=9,
        color=COLORS['muted']
    )


add_data_box(
    0.4, 2.4, 3.6, 1.8,
    'ข้อมูล e-GP ทั้งหมด',
    source_record_count,
    'ระเบียน',
    COLORS['primary']
)
add_data_box(
    5.7, 2.4, 3.6, 1.8,
    'เลือกสัญญาจ้างก่อสร้าง',
    construction_contract_count,
    'สัญญา',
    COLORS['secondary']
)
add_data_box(
    11.0, 2.4, 3.6, 1.8,
    'ข้อมูลระดับสัญญา',
    analysis_contract_count,
    'สัญญา',
    COLORS['highlight']
)

add_data_arrow((4.0, 3.3), (5.7, 3.3), 'เลือกประเภทจ้างก่อสร้าง')
add_data_arrow((9.3, 3.3), (11.0, 3.3), 'ตัดรหัสผู้รับจ้างขึ้นต้น D\n363 สัญญา')

ax.text(
    0.4,
    6.25,
    'เส้นทางการเตรียมข้อมูลระดับสัญญา',
    fontsize=14,
    fontweight='semibold',
    color=COLORS['text'],
    ha='left'
)
ax.text(
    0.4,
    5.78,
    'Grain: รหัสโครงการ + รหัสผู้รับจ้าง + เลขที่สัญญา',
    fontsize=9,
    color=COLORS['muted'],
    ha='left'
)

fig.tight_layout()
save_figure(fig, 'fig05_01_data_cleaning_journey')
plt.show()

data_journey = pd.DataFrame({
    'ขั้นตอน': [
        'ข้อมูล e-GP ทั้งหมด',
        'สัญญาจ้างก่อสร้าง',
        'ตัดรหัสผู้รับจ้างขึ้นต้น D',
        'ข้อมูลระดับสัญญาที่ใช้วิเคราะห์'
    ],
    'จำนวน': [
        source_record_count,
        construction_contract_count,
        excluded_d_count,
        analysis_contract_count
    ],
    'หน่วย': [
        'ระเบียน',
        'สัญญา',
        'สัญญาที่ตัดออก',
        'สัญญา'
    ]
})

display_table(data_journey)


## Story 2 — ภาพรวมสัญญาก่อสร้างบอกอะไร

เริ่มจากสำรวจสัญญาก่อสร้างทั้งหมดก่อนกำหนดขอบเขต Pattern โดยดูการกระจายตามช่วงวงเงินควบคู่กับสัดส่วนวิธีจัดซื้อ ทุกแท่งแสดงทั้งจำนวนสัญญาและสัดส่วนจากสัญญาก่อสร้างทั้งหมด


In [ ]:
budget_bins = [
    -np.inf,
    100000,
    200000,
    300000,
    400000,
    500000,
    1000000,
    5000000,
    10000000,
    50000000,
    np.inf
]
budget_labels = [
    '≤100K',
    '100–200K',
    '200–300K',
    '300–400K',
    '400–500K',
    '500K–1M',
    '1–5M',
    '5–10M',
    '10–50M',
    '>50M'
]

budget_band = pd.cut(
    contract_data[contract_value_column],
    bins=budget_bins,
    labels=budget_labels,
    right=True,
    include_lowest=True
)

budget_summary = (
    budget_band
    .value_counts(sort=False)
    .rename('จำนวนสัญญา')
    .reset_index()
    .rename(columns={contract_value_column: 'ช่วงวงเงิน', 'index': 'ช่วงวงเงิน'})
)
budget_summary['สัดส่วน (%)'] = (
    budget_summary['จำนวนสัญญา'] /
    budget_summary['จำนวนสัญญา'].sum() * 100
)

method_overview = (
    contract_data[method_column]
    .fillna('ไม่ระบุ')
    .value_counts()
    .rename('จำนวนสัญญา')
    .reset_index()
    .rename(columns={'index': method_column})
)
method_overview['สัดส่วน (%)'] = (
    method_overview['จำนวนสัญญา'] /
    method_overview['จำนวนสัญญา'].sum() * 100
)
method_overview['วิธีจัดซื้อ'] = (
    method_overview[method_column]
    .replace({
        'ประกวดราคาอิเล็กทรอนิกส์ (e-bidding)': 'e-bidding'
    })
)

fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE)
ax_budget, ax_method = axes

budget_plot = budget_summary.iloc[::-1].reset_index(drop=True)
budget_colors = [
    COLORS['highlight'] if label == '400–500K'
    else COLORS['primary']
    for label in budget_plot['ช่วงวงเงิน'].astype(str)
]
budget_bars = ax_budget.barh(
    budget_plot['ช่วงวงเงิน'].astype(str),
    budget_plot['จำนวนสัญญา'],
    color=budget_colors,
    height=0.62
)
budget_value_labels = [
    f'{count:,.0f}  |  {share:.2f}%'
    for count, share in zip(
        budget_plot['จำนวนสัญญา'],
        budget_plot['สัดส่วน (%)']
    )
]
ax_budget.bar_label(
    budget_bars,
    labels=budget_value_labels,
    padding=4,
    fontsize=8,
    color=COLORS['text']
)
ax_budget.set_title('จำนวนสัญญาตามช่วงวงเงิน', pad=22)
add_subtitle(ax_budget, 'Label แสดงจำนวนสัญญาและสัดส่วนจากทั้งหมด')
ax_budget.set_xlabel('จำนวนสัญญา')
ax_budget.set_ylabel('')
ax_budget.set_xlim(
    0,
    budget_plot['จำนวนสัญญา'].max() * 1.42
)
clean_axis(ax_budget, grid_axis='x')

method_plot = method_overview.sort_values('จำนวนสัญญา', ascending=False)
method_colors = [
    COLORS['highlight'] if method == 'เฉพาะเจาะจง'
    else COLORS['secondary'] if method == 'e-bidding'
    else COLORS['neutral']
    for method in method_plot['วิธีจัดซื้อ']
]
wedges, _ = ax_method.pie(
    method_plot['จำนวนสัญญา'],
    colors=method_colors,
    startangle=90,
    counterclock=False,
    wedgeprops={'width': 0.46, 'edgecolor': 'white'}
)
method_total = method_plot['จำนวนสัญญา'].sum()
ax_method.text(
    0,
    0.08,
    f'{method_total:,.0f}',
    ha='center',
    va='center',
    fontsize=13,
    fontweight='semibold',
    color=COLORS['text']
)
ax_method.text(
    0,
    -0.12,
    'สัญญา',
    ha='center',
    va='center',
    fontsize=8,
    color=COLORS['muted']
)
method_legend = [
    f'{method}: {count:,.0f} ({share:.2f}%)'
    for method, count, share in zip(
        method_plot['วิธีจัดซื้อ'],
        method_plot['จำนวนสัญญา'],
        method_plot['สัดส่วน (%)']
    )
]
ax_method.legend(
    wedges,
    method_legend,
    frameon=False,
    fontsize=8,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.16)
)
ax_method.set_title('สัดส่วนสัญญาตามวิธีจัดซื้อ', pad=22)

fig.subplots_adjust(
    left=0.12,
    right=0.96,
    top=0.82,
    bottom=0.20,
    wspace=0.34
)
save_figure(fig, 'fig05_02_construction_overview')
plt.show()


## Story 3 — บริเวณ 500,000 บาทเกิดอะไรขึ้น และสัมพันธ์กับวิธีใด

จากภาพรวมพบว่าช่วง 400,000–500,000 บาทมีจำนวนสัญญาสูง จึงขยายช่วง 400,000–550,000 บาท และใช้ stacked column แยกตามวิธีจัดซื้อ เพื่อดูพร้อมกันทั้งการกระจุกของสัญญาและองค์ประกอบของแต่ละช่วงวงเงิน


In [ ]:
focus_data = contract_data.loc[
    contract_data[contract_value_column].between(400000, 550000)
].copy()

bin_edges = np.arange(400000, 560000, 10000)
focus_data['ช่วงวงเงิน'] = pd.cut(
    focus_data[contract_value_column],
    bins=bin_edges,
    right=True,
    include_lowest=True
)

focus_data['วิธีจัดซื้อ'] = (
    focus_data[method_column]
    .fillna('ไม่ระบุ')
    .replace({
        'ประกวดราคาอิเล็กทรอนิกส์ (e-bidding)': 'e-bidding'
    })
)

stacked_summary = (
    focus_data
    .groupby(
        ['ช่วงวงเงิน', 'วิธีจัดซื้อ'],
        observed=False
    )
    .size()
    .unstack(fill_value=0)
)

method_order = (
    stacked_summary
    .sum()
    .sort_values(ascending=False)
    .index
)
stacked_summary = stacked_summary[method_order]
stacked_summary.index = [
    f'{int(interval.left / 1000):,}–{int(interval.right / 1000):,}'
    for interval in stacked_summary.index
]

stack_colors = [
    COLORS['highlight'] if method == 'เฉพาะเจาะจง'
    else COLORS['secondary'] if method == 'e-bidding'
    else COLORS['neutral']
    for method in stacked_summary.columns
]

fig, ax = plt.subplots(figsize=FIG_SIZE)

stacked_summary.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=stack_colors,
    width=0.76,
    edgecolor='white',
    linewidth=0.4
)

totals = stacked_summary.sum(axis=1)
maximum_total = totals.max()

for position, total in enumerate(totals):
    ax.text(
        position,
        total + maximum_total * 0.018,
        f'{total:,.0f}',
        ha='center',
        va='bottom',
        fontsize=8,
        color=COLORS['text']
    )

near_position = list(stacked_summary.index).index('490–500')
ax.axvspan(
    near_position - 0.48,
    near_position + 0.48,
    color=COLORS['risk'],
    alpha=0.07,
    zorder=0
)
ax.axvline(
    near_position + 0.5,
    color=COLORS['risk'],
    linewidth=1.2,
    linestyle='--'
)
ax.text(
    near_position + 0.62,
    maximum_total * 1.10,
    '500,000 บาท',
    color=COLORS['risk'],
    fontsize=8,
    va='center'
)

ax.set_title('จำนวนสัญญารอบ 500,000 บาท แยกตามวิธีจัดซื้อ', pad=22)
add_subtitle(ax, 'ช่วงละ 10,000 บาท; ตัวเลขเหนือแท่งแสดงจำนวนสัญญารวม')
ax.set_xlabel('ช่วงวงเงินสัญญา (พันบาท)')
ax.set_ylabel('จำนวนสัญญา')
ax.tick_params(axis='x', rotation=45)
ax.set_ylim(0, maximum_total * 1.20)
ax.legend(
    title='วิธีจัดซื้อ',
    frameon=False,
    fontsize=8,
    title_fontsize=8,
    ncol=min(3, len(stacked_summary.columns)),
    loc='upper left'
)
clean_axis(ax, grid_axis='y')

fig.subplots_adjust(
    left=0.10,
    right=0.96,
    top=0.82,
    bottom=0.22
)
save_figure(fig, 'fig05_03_near_500k_by_method')
plt.show()

near_method_summary = (
    focus_data.loc[
        focus_data[contract_value_column].between(490000, 500000)
    ]
    .groupby('วิธีจัดซื้อ')
    .size()
    .rename('จำนวนสัญญา')
    .reset_index()
)
near_method_summary['สัดส่วน (%)'] = (
    near_method_summary['จำนวนสัญญา'] /
    near_method_summary['จำนวนสัญญา'].sum() * 100
)
display_table(
    near_method_summary
    .sort_values('จำนวนสัญญา', ascending=False)
    .reset_index(drop=True)
)


## Story 4 — จากข้อค้นพบ จึงกำหนดขอบเขตศึกษา Pattern

จากสัญญาก่อสร้างทั้งหมด เราลดขอบเขตตามวงเงินก่อน แล้วจึงเลือกวิธีเฉพาะเจาะจง ขอบเขตสุดท้ายสำหรับวิเคราะห์ Pattern คือ **วงเงินไม่เกิน 500,000 บาทและใช้วิธีเฉพาะเจาะจง** ส่วนช่วงใกล้เพดาน 490,000–500,000 บาทจะถูกใช้ภายหลังใน Pattern 1 ไม่ใช่เงื่อนไขของ Study Scope


In [ ]:
under_ceiling_data = contract_data.loc[
    contract_data[contract_value_column].le(500000)
].copy()

scope_story = pd.DataFrame({
    'ขอบเขต': [
        'สัญญาก่อสร้างทั้งหมด',
        'วงเงิน ≤500,000 บาท',
        'วงเงิน ≤500,000 บาท และวิธีเฉพาะเจาะจง'
    ],
    'จำนวนสัญญา': [
        len(contract_data),
        len(under_ceiling_data),
        len(study_data)
    ],
    'สี': [
        COLORS['primary'],
        COLORS['secondary'],
        COLORS['highlight']
    ]
})

scope_story['สัดส่วนจากทั้งหมด (%)'] = (
    scope_story['จำนวนสัญญา'] /
    len(contract_data) * 100
)

plot_data = scope_story.iloc[::-1].reset_index(drop=True)
y_position = np.arange(len(plot_data))

fig, ax = plt.subplots(figsize=FIG_SIZE)

ax.barh(
    y_position,
    [100] * len(plot_data),
    color=COLORS['light'],
    height=0.34,
    edgecolor='none'
)

for y, share, count, color, label in zip(
    y_position,
    plot_data['สัดส่วนจากทั้งหมด (%)'],
    plot_data['จำนวนสัญญา'],
    plot_data['สี'],
    plot_data['ขอบเขต']
):
    ax.barh(
        y,
        share,
        color=color,
        height=0.34,
        edgecolor='none'
    )
    ax.text(
        0,
        y + 0.31,
        label,
        ha='left',
        va='bottom',
        fontsize=10,
        color=COLORS['text'],
        fontweight='semibold'
    )
    ax.text(
        100,
        y + 0.31,
        f'{count:,.0f} สัญญา  |  {share:.2f}%',
        ha='right',
        va='bottom',
        fontsize=9,
        color=color,
        fontweight='semibold'
    )

ax.set_title('ขอบเขตข้อมูลสำหรับวิเคราะห์ Pattern', pad=24)
add_subtitle(ax, 'ลดขอบเขตตามวงเงิน แล้วเลือกเฉพาะวิธีเฉพาะเจาะจง')
ax.set_xlim(0, 100)
ax.set_ylim(-0.65, len(plot_data) - 0.25)
ax.set_xlabel('สัดส่วนจากสัญญาก่อสร้างทั้งหมด (%)')
ax.set_yticks([])
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.grid(axis='x', color=COLORS['light'])
ax.grid(axis='y', visible=False)

fig.subplots_adjust(
    left=0.10,
    right=0.94,
    top=0.80,
    bottom=0.17
)
save_figure(fig, 'fig05_04_pattern_study_scope')
plt.show()

display_table(
    scope_story[[
        'ขอบเขต',
        'จำนวนสัญญา',
        'สัดส่วนจากทั้งหมด (%)'
    ]]
)


## Story 5 — Pattern 1: คู่หน่วยงาน–ผู้รับจ้างใดมีสัญญาใกล้เพดานเกิดซ้ำ

ภายในขอบเขตศึกษา เปลี่ยน grain จากสัญญาเดี่ยวเป็นคู่ **ชื่อหน่วยงาน + ผู้รับจ้าง** แล้วจัดอันดับด้วยจำนวนสัญญาใกล้เพดาน ส่วน label แสดงสัดส่วนสัญญาใกล้เพดานภายในคู่เดียวกัน


In [ ]:
top_repeated_pairs = (
    repeated_pairs
    .sort_values(
        ['จำนวนสัญญาใกล้เพดาน', 'สัดส่วนจำนวนใกล้เพดาน (%)'],
        ascending=False
    )
    .head(10)
    .copy()
)

top_repeated_pairs['คู่หน่วยงาน–ผู้รับจ้าง'] = [
    shorten(
        f'{agency} | {supplier}',
        width=46,
        placeholder='…'
    )
    for agency, supplier in zip(
        top_repeated_pairs[agency_column],
        top_repeated_pairs[supplier_name_column]
    )
]

plot_data = top_repeated_pairs.iloc[::-1]

fig, ax = plt.subplots(figsize=FIG_SIZE)

bars = ax.barh(
    plot_data['คู่หน่วยงาน–ผู้รับจ้าง'],
    plot_data['จำนวนสัญญาใกล้เพดาน'],
    color=COLORS['primary'],
    height=0.60
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f}'
        for count in plot_data['จำนวนสัญญาใกล้เพดาน']
    ],
    padding=5,
    fontsize=9,
    color=COLORS['text']
)

ax.set_title('คู่หน่วยงาน–ผู้รับจ้างที่มีสัญญาใกล้เพดานมากที่สุด', pad=24)
add_subtitle(ax, '10 อันดับแรก; label แสดงจำนวนสัญญาใกล้เพดาน')
ax.set_xlabel('จำนวนสัญญาใกล้เพดาน')
ax.set_ylabel('')
ax.set_xlim(
    0,
    plot_data['จำนวนสัญญาใกล้เพดาน'].max() * 1.18
)
clean_axis(ax, grid_axis='x')

fig.subplots_adjust(
    left=0.40,
    right=0.96,
    top=0.83,
    bottom=0.12
)
save_figure(fig, 'fig05_05_pattern1_top10_repeated_pairs')
plt.show()


## Story 6 — Pattern 2: คู่หน่วยงาน–ผู้รับจ้างใดมีจำนวนสัญญาสูง

Pattern 2 คัดคู่ที่ผู้รับจ้างมีอย่างน้อย 10 สัญญา และครองสัดส่วนทั้งจำนวนสัญญาและมูลค่าในหน่วยงานตั้งแต่ 80% ขึ้นไป เนื่องจากหลายคู่มีสัดส่วน 100% visual จึงจัดอันดับด้วย **จำนวนสัญญา** เพื่อให้เห็นขนาดของการพึ่งพา


In [ ]:
top_dependency_pairs = (
    pattern2_pairs
    .sort_values(
        [
            'จำนวนรายการของผู้รับจ้าง',
            'มูลค่าของผู้รับจ้าง (บาท)'
        ],
        ascending=False
    )
    .head(10)
    .copy()
)

top_dependency_pairs['คู่หน่วยงาน–ผู้รับจ้าง'] = [
    shorten(
        f'{agency} | {supplier}',
        width=48,
        placeholder='…'
    )
    for agency, supplier in zip(
        top_dependency_pairs[agency_column],
        top_dependency_pairs['ผู้รับจ้าง']
    )
]

plot_data = top_dependency_pairs.iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=FIG_SIZE)

bars = ax.barh(
    plot_data['คู่หน่วยงาน–ผู้รับจ้าง'],
    plot_data['จำนวนรายการของผู้รับจ้าง'],
    color=COLORS['primary'],
    height=0.62
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f} สัญญา'
        for count in plot_data['จำนวนรายการของผู้รับจ้าง']
    ],
    padding=5,
    fontsize=9,
    color=COLORS['text']
)

ax.set_title('คู่หน่วยงาน–ผู้รับจ้างที่มีจำนวนสัญญาสูงใน Pattern 2', pad=24)
add_subtitle(ax, '10 อันดับแรก เรียงตามจำนวนสัญญาของผู้รับจ้างในหน่วยงาน')
ax.set_xlabel('จำนวนสัญญา')
ax.set_ylabel('')
ax.set_xlim(
    0,
    plot_data['จำนวนรายการของผู้รับจ้าง'].max() * 1.30
)
clean_axis(ax, grid_axis='x')

fig.subplots_adjust(left=0.40, right=0.96, top=0.83, bottom=0.12)
save_figure(fig, 'fig05_06_pattern2_top10_contract_count')
plt.show()

pattern2_table = (
    top_dependency_pairs[[
        agency_column,
        'ผู้รับจ้าง',
        'จำนวนรายการของผู้รับจ้าง',
        'มูลค่าของผู้รับจ้าง (บาท)'
    ]]
    .rename(columns={
        'จำนวนรายการของผู้รับจ้าง': 'จำนวนสัญญา',
        'มูลค่าของผู้รับจ้าง (บาท)': 'มูลค่าสัญญารวม (บาท)'
    })
)

display_table(pattern2_table)


## Story 7 — จากสัญญาก่อสร้างทั้งหมดสู่จุดตัดของสอง Pattern

ภาพนี้สรุปเส้นทางทั้งหมดอีกครั้ง: เริ่มจากสัญญาก่อสร้างทั้งหมด ลดเหลือขอบเขตศึกษา จากนั้นคำนวณ Pattern 1 และ Pattern 2 แยกกัน ก่อนเลือกเฉพาะสัญญาที่เข้าเงื่อนไขทั้งสองพร้อมกันเป็น Priority Review


In [ ]:
all_contract_count = len(contract_data)
study_count = len(study_data)
pattern1_count = int(contract_flags['flag_pattern_1'].sum())
pattern2_count = int(contract_flags['flag_pattern_2'].sum())
priority_count = int(contract_flags['priority_review'].sum())

fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.set_xlim(0, 15)
ax.set_ylim(0, 8)
ax.axis('off')


def add_recap_box(x, y, width, height, title, value, color):
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle='round,pad=0.03,rounding_size=0.14',
        linewidth=1.2,
        edgecolor=color,
        facecolor='white'
    )
    ax.add_patch(box)
    ax.text(
        x + width / 2,
        y + height * 0.66,
        title,
        ha='center',
        va='center',
        fontsize=9,
        color=COLORS['text'],
        linespacing=1.15
    )
    ax.text(
        x + width / 2,
        y + height * 0.28,
        f'{value:,.0f}',
        ha='center',
        va='center',
        fontsize=13,
        fontweight='semibold',
        color=color
    )


def add_recap_arrow(start, end, color=COLORS['neutral']):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle='-|>',
            mutation_scale=15,
            linewidth=1.4,
            color=color,
            connectionstyle='arc3,rad=0.0'
        )
    )


add_recap_box(
    0.35, 3.2, 2.5, 1.6,
    'สัญญาก่อสร้าง\nทั้งหมด',
    all_contract_count,
    COLORS['primary']
)
add_recap_box(
    3.55, 3.2, 2.5, 1.6,
    'Scope\nเฉพาะเจาะจง ≤500K',
    study_count,
    COLORS['secondary']
)
add_recap_box(
    6.85, 5.15, 2.7, 1.6,
    'Pattern 1\nใกล้เพดานเกิดซ้ำ',
    pattern1_count,
    COLORS['highlight']
)
add_recap_box(
    6.85, 1.25, 2.7, 1.6,
    'Pattern 2\nพึ่งพาผู้รับจ้างสูง',
    pattern2_count,
    COLORS['primary']
)
add_recap_box(
    11.0, 3.2, 3.3, 1.6,
    'Pattern 1 & 2\nPriority Review',
    priority_count,
    COLORS['risk']
)

add_recap_arrow((2.85, 4.0), (3.55, 4.0))
add_recap_arrow((6.05, 4.15), (6.85, 5.90))
add_recap_arrow((6.05, 3.85), (6.85, 2.05))
add_recap_arrow((9.55, 5.90), (11.0, 4.45), COLORS['highlight'])
add_recap_arrow((9.55, 2.05), (11.0, 3.55), COLORS['primary'])

ax.text(
    0.35,
    7.55,
    'เส้นทางจากสัญญาก่อสร้างทั้งหมดสู่ Priority Review',
    fontsize=14,
    fontweight='semibold',
    color=COLORS['text'],
    ha='left'
)
ax.text(
    0.35,
    7.10,
    'Pattern 1 และ Pattern 2 คำนวณแยกกัน แล้วเลือกเฉพาะจุดตัดของสองเงื่อนไข',
    fontsize=9,
    color=COLORS['muted'],
    ha='left'
)

fig.subplots_adjust(
    left=0.04,
    right=0.96,
    top=0.92,
    bottom=0.05
)
save_figure(fig, 'fig05_07_pattern_intersection_journey')
plt.show()


## Story 8 — คู่หน่วยงาน–ผู้รับจ้างใดควรเปิดเอกสารก่อน

จัดอันดับสัญญาที่เข้า Pattern 1 และ Pattern 2 พร้อมกันตามจำนวนสัญญาของคู่หน่วยงาน–ผู้รับจ้าง เพื่อกำหนดลำดับการเปิดเอกสาร ไม่ใช่จัดอันดับความทุจริต ตารางใต้ภาพใช้ Top 10 ชุดเดียวกับ visual


In [ ]:
priority_pair_summary = (
    priority_contracts
    .groupby(
        [agency_column, supplier_id_column],
        dropna=False
    )
    .agg({
        supplier_name_column: 'first',
        province_column: 'first',
        contract_column: 'size',
        contract_value_column: 'sum'
    })
    .reset_index()
    .rename(columns={
        contract_column: 'จำนวนสัญญาตรวจสอบ',
        contract_value_column: 'มูลค่ารวม (บาท)'
    })
    .sort_values(
        ['จำนวนสัญญาตรวจสอบ', 'มูลค่ารวม (บาท)'],
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

priority_pair_summary.insert(
    0,
    'ลำดับ',
    np.arange(1, len(priority_pair_summary) + 1)
)

priority_pair_summary['คู่หน่วยงาน–ผู้รับจ้าง'] = [
    shorten(
        f'{agency} | {supplier}',
        width=48,
        placeholder='…'
    )
    for agency, supplier in zip(
        priority_pair_summary[agency_column],
        priority_pair_summary[supplier_name_column]
    )
]

plot_data = priority_pair_summary.iloc[::-1]

fig, ax = plt.subplots(figsize=FIG_SIZE)

bars = ax.barh(
    plot_data['คู่หน่วยงาน–ผู้รับจ้าง'],
    plot_data['จำนวนสัญญาตรวจสอบ'],
    color=COLORS['risk'],
    height=0.64
)

labels = [
    f'{count:,.0f} สัญญา  |  {value / 1_000_000:,.1f} ล้านบาท'
    for count, value in zip(
        plot_data['จำนวนสัญญาตรวจสอบ'],
        plot_data['มูลค่ารวม (บาท)']
    )
]

ax.bar_label(
    bars,
    labels=labels,
    padding=6,
    fontsize=9,
    color=COLORS['text']
)

ax.set_title('คู่หน่วยงาน–ผู้รับจ้างในรายการตรวจสอบลำดับแรก', pad=28)
add_subtitle(ax, '10 อันดับแรก; label แสดงจำนวนรายการและมูลค่ารวม')
ax.set_xlabel('จำนวนสัญญา')
ax.set_ylabel('')
ax.set_xlim(0, plot_data['จำนวนสัญญาตรวจสอบ'].max() * 1.72)
clean_axis(ax, grid_axis='x')

fig.subplots_adjust(left=0.40, right=0.96, top=0.83, bottom=0.12)
save_figure(fig, 'fig05_08_priority_top10_pairs')
plt.show()

priority_table = priority_pair_summary[[
    'ลำดับ',
    agency_column,
    supplier_name_column,
    province_column,
    'จำนวนสัญญาตรวจสอบ',
    'มูลค่ารวม (บาท)'
]].copy()

display_table(priority_table)


## สรุปลำดับการใช้ภาพ

| ขั้นตอน | Figure | คำถามที่ภาพตอบ |
|---:|---|---|
| 1 | `fig05_01_data_cleaning_journey` | ข้อมูล 3,964,924 ระเบียนเหลือข้อมูลระดับสัญญา 179,716 สัญญาอย่างไร |
| 2 | `fig05_02_construction_overview` | สัญญาก่อสร้างกระจายตามวงเงินและวิธีจัดซื้ออย่างไร |
| 3 | `fig05_03_near_500k_by_method` | บริเวณ 500,000 บาทมีการกระจุกหรือไม่ และเกิดจากวิธีใด |
| 4 | `fig05_04_pattern_study_scope` | สัญญาก่อสร้างทั้งหมดลดเหลือ Scope สำหรับ Pattern อย่างไร |
| 5 | `fig05_05_pattern1_top10_repeated_pairs` | คู่ใดมีสัญญาใกล้เพดานเกิดซ้ำมาก |
| 6 | `fig05_06_pattern2_top10_contract_count` | คู่ที่เข้า Pattern 2 คู่ใดมีจำนวนสัญญามาก |
| 7 | `fig05_07_pattern_intersection_journey` | ข้อมูลทั้งหมดลดลงสู่ Scope, Pattern 1, Pattern 2 และจุดตัดอย่างไร |
| 8 | `fig05_08_priority_top10_pairs` | คู่ใดควรเปิดเอกสารตรวจสอบก่อน |

ภาพทั้งหมดใช้ canvas 12 × 6.75 นิ้ว ที่ 120 DPI หรือ 1,440 × 810 pixels ซึ่งเหมาะกับการอ่านบน GitHub และกำหนด margin ภายในแต่ละภาพเพื่อป้องกันข้อความตกขอบ
